# 07 — EMTboost HPO · Zhou·Qian·Yang (2019)
**"Tweedie Gradient Boosting for Extremely Unbalanced Zero-inflated Data"** (arXiv:1811.10192) 충실 구현 모델을 Optuna로 튜닝한다.

## 모델 (논문 그대로 — `modules.zit_EMT.EMTboost`)
- 혼합: `Y=0` (확률 1−q) / `Y ~ Tweedie(μ,φ)` (확률 q). **q·φ는 전역 스칼라**, μ만 LightGBM Tweedie(f(X)).
- 예측: **E[Y|x] = q·μ(x)** (논문 eq.41).
- EM: E-step(eq.14) → M-step μ(eq.21, weight=δ₁) / φ(eq.23 golden section) / q(eq.24 = mean δ₁).

## die ↔ unit 처리 (bag 아님 — `zit_only` 방식)
- unit health(target)을 그 unit의 **4 die에 그대로 복사**해 die-level로 학습 (`y_die = unit_y.map(...)`).
- die별 예측 `q·μ` → **MEAN**으로 unit 집계 (`groupby(ufs_serial).mean()`).
- CV는 **`ufs_serial`(unit) 단위 5-fold** — 같은 unit의 4 die가 train/val에 섞이지 않게.
- **tau_pi 게이트 없음**: EMT의 pi(=1−q)는 전 die 동일한 스칼라라 die-gate가 퇴화 → 논문 그대로 q·μ.

## 전처리 (최종과 동일)
- **현재 전처리 모듈**(`modules.preprocess` → `2_preprocessing/cleaning.py`)을 그대로 사용. 표준 impute가 이미 spatial→lot median→global median.
- `PP_FIXED`(missing 0.3 / corr 0.9 / indicator 0.05 / spatial 6.0 / post_corr 0.96), `CLIP_Y_EXTREME`, `add_meta_features(position_mode='raw', use_die_xy=True)` → n_features=576.
- (옵션) `USE_PRECOMPUTED=True`면 `pp.npy` 캐시로 즉시 로드 — 표준 모듈 출력과 byte-identical.

## 튜닝 예산
- `END_AT`(절대 시각, 기본=내일 오전) 또는 `TIMEOUT_HOURS`로 제어. 한 노트북 프로세스가 마감까지 계속 trial 소화.

> ⚠ **Colab**: `3_modeling/modules/zit_EMT.py`가 새로 추가됨 → `modeling.zip` 재업로드해야 import 됨.


## 0. 환경 설정

In [1]:
from pathlib import Path
import gc, json, os, runpy, sys, time
from datetime import datetime

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for cand in (p, *p.parents):
        if (cand / 'setup.py').exists() and (cand / 'utils').exists():
            return cand
    raise RuntimeError('프로젝트 루트를 찾지 못함: setup.py + utils/ 기준')


ROOT = find_project_root()
runpy.run_path(str(ROOT / 'setup.py'))   # 경로/폰트/requirements 부트스트랩 (Colab/Local 공통)

from utils.config import (
    PROJECT_ROOT as CFG_PROJECT_ROOT,
    OUTPUT_DIR, DATA_DIR,
    TARGET_COL, KEY_COL, DIE_KEY_COL,
    SEED as DEFAULT_SEED,
)
from utils.data import load_all, get_feat_cols, split_xs

PP_DIR = Path(CFG_PROJECT_ROOT) / '2_preprocessing'
if str(PP_DIR) not in sys.path:
    sys.path.insert(0, str(PP_DIR))
MOD_DIR = Path(CFG_PROJECT_ROOT) / '3_modeling'
if str(MOD_DIR) not in sys.path:
    sys.path.insert(0, str(MOD_DIR))

from meta_features import add_meta_features
from modules import hpo, preprocess
from modules.zit_EMT import EMTboost   # 논문 충실 EMTboost (scalar q/φ, μ=LightGBM Tweedie)

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from optuna.storages import RDBStorage, RetryFailedTrialCallback
from sklearn.model_selection import KFold

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)
optuna.logging.set_verbosity(optuna.logging.WARNING)

PROJECT_ROOT = Path(CFG_PROJECT_ROOT)
OUTPUT_DIR = Path(OUTPUT_DIR)
DATA_DIR = Path(DATA_DIR)
print('optuna', optuna.__version__)
print('PROJECT_ROOT', PROJECT_ROOT)


setup 완료
optuna 4.7.0
PROJECT_ROOT C:\Users\Dell5371\Desktop\기업연계프로젝트


## 1. 실행 설정

In [2]:
EXP_ID = 'emt-boost-hp-001'
USER = 'jh'

# RESUME: 기존 study db에 이어서 돌릴지. (Local에서 같은 EXP_ID db가 있는데 RESUME=False면
#         DuplicatedStudyError → 처음부터 다시 하려면 해당 optuna_*.db를 먼저 지운다.)
RESUME = False

SEED = int(DEFAULT_SEED)
N_FOLDS = 5
# 단일 노트북 프로세스라 LightGBM에 물리코어를 넉넉히 준다. 큰 PC면 그대로, 작은 PC면 줄인다.
N_JOBS = max(1, (os.cpu_count() or 4))

# ── 튜닝 예산: 내일 오전까지 ──────────────────────────────────────────────
# END_AT(절대 시각) 우선. 시작 시점에 따라 남은 시간이 자동 계산된다. None이면 TIMEOUT_HOURS 사용.
END_AT = '2026-06-07T08:00'    # 예: 내일 오전 8시. 지났거나 None이면 TIMEOUT_HOURS로 대체.
TIMEOUT_HOURS = 10.0           # END_AT=None일 때의 벽시계 제한.
N_TRIALS = 1_000_000           # 사실상 무제한 — timeout이 먼저 끊는다.
N_STARTUP_TRIALS = 20          # 작은 탐색공간(11 HP) + warm-start anchor → 적당히.

CLIP_Y_EXTREME = True          # 최종과 동일: train health 1.0 → 두 번째 최댓값으로 clip.

# 전처리: 기본은 '현재 전처리 모듈'로 자가전처리. True면 pp.npy 캐시(=표준 모듈과 byte-identical) 즉시 로드.
USE_PRECOMPUTED = False
PRECOMPUTED_DIR = DATA_DIR / 'precomputed' / 'bag_zit_pp'

# 산출물 (db: 4_output/01_zit/emt_boost/hp/<exp 끝 숫자>/)
OUT_DIR = OUTPUT_DIR / '01_zit' / 'emt_boost' / 'hp' / EXP_ID.split('-')[-1]
OUT_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = OUT_DIR / f'optuna_{USER}_{EXP_ID}.db'
print('OUT_DIR', OUT_DIR)
print('DB_PATH', DB_PATH)
print(f'N_JOBS={N_JOBS}  N_FOLDS={N_FOLDS}  END_AT={END_AT}  TIMEOUT_HOURS={TIMEOUT_HOURS}')

# 최종과 동일한 고정 전처리 파라미터
PP_FIXED = {
    'missing_threshold': 0.30, 'corr_threshold': 0.90, 'corr_keep_by': 'std',
    'add_indicator': True, 'indicator_threshold': 0.05, 'spatial_max_dist': 6.0,
    'post_impute_corr_threshold': 0.96, 'post_impute_corr_keep_by': 'std',
}

# ── EMTboost 전용 탐색공간 ────────────────────────────────────────────────
# μ(LightGBM Tweedie) HP 9개 + EM축(zeta, n_em_iters). π/φ는 스칼라라 GBM HP 없음. tau_pi 없음.
# μ ranges/anchor는 같은 Tweedie μ-GBM을 튜닝한 zit_only(hp/004)에서 가져와 살짝 넓힘. zeta/n_em은 EM에 여유.
EMT_SEARCH = {
    'zeta':                 {'type': 'float', 'low': 1.01,  'high': 1.25,  'log': False},
    'n_em_iters':           {'type': 'int',   'low': 10,    'high': 30},
    'mu_n_estimators':      {'type': 'int',   'low': 120,   'high': 280},
    'mu_learning_rate':     {'type': 'float', 'low': 9e-4,  'high': 5e-3,  'log': True},
    'mu_num_leaves':        {'type': 'int',   'low': 16,    'high': 160},
    'mu_max_depth':         {'type': 'int',   'low': 2,     'high': 8},
    'mu_min_child_samples': {'type': 'int',   'low': 120,   'high': 340},
    'mu_subsample':         {'type': 'float', 'low': 0.55,  'high': 0.90,  'log': False},
    'mu_colsample_bytree':  {'type': 'float', 'low': 0.12,  'high': 0.40,  'log': False},
    'mu_reg_alpha':         {'type': 'float', 'low': 5e-5,  'high': 1e-2,  'log': True},
    'mu_reg_lambda':        {'type': 'float', 'low': 3e-5,  'high': 1e-2,  'log': True},
}

# warm-start anchor: zit-only-final-003 best #24 의 μ + zeta/n_em (같은 Tweedie μ-GBM). 전부 위 range 안.
EMT_ANCHOR = {
    'zeta': 1.1200196096433799,
    'n_em_iters': 16,
    'mu_n_estimators': 206,
    'mu_learning_rate': 0.001544804994892871,
    'mu_num_leaves': 29,
    'mu_max_depth': 3,
    'mu_min_child_samples': 269,
    'mu_subsample': 0.8129218946739942,
    'mu_colsample_bytree': 0.18344245348572596,
    'mu_reg_alpha': 0.001211879891858438,
    'mu_reg_lambda': 0.003235224001159518,
}


OUT_DIR C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\01_zit\emt_boost\hp\001
DB_PATH C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\01_zit\emt_boost\hp\001\optuna_jh_emt-boost-hp-001.db
N_JOBS=20  N_FOLDS=5  END_AT=2026-06-07T08:00  TIMEOUT_HOURS=10.0


## 2. 데이터 로드 (최종과 동일 전처리)

In [3]:
def _load_self_preprocess():
    # 현재 전처리 모듈로 자가전처리 (zit_only/bag worker의 load_preprocessed_data와 동일 절차)
    xs, ys = load_all()
    feat_cols = get_feat_cols(xs)
    xs_dict = split_xs(xs)

    ys_input = {k: v.copy() for k, v in ys.items()}
    if CLIP_Y_EXTREME:
        y_raw = ys_input['train'][TARGET_COL]
        second_max = y_raw[y_raw < y_raw.max()].max()
        n_clipped = int((y_raw >= 1.0).sum())
        ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
        print(f'[CLIP_Y_EXTREME] 1.0 -> {second_max:.6f}, n={n_clipped}')

    pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PP_FIXED)
    xs_train, xs_val, xs_test = pp['xs_train'], pp['xs_val'], pp['xs_test']
    feat_cols_clean = pp['feat_cols']
    feat_cols_clean = add_meta_features(
        xs_train, xs_val, xs_test, feat_cols_clean,
        position_mode='raw', use_die_xy=True,
    )

    x_train = xs_train[feat_cols_clean].values.astype(np.float64)
    uid_train_die = xs_train[KEY_COL].values
    y_train_unit_s = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
    # ★ bag 아님: unit health를 그 unit의 4 die에 그대로 복사 (broadcast)
    y_train_die = xs_train[KEY_COL].map(y_train_unit_s).values.astype(np.float64)
    print(f'[self-preprocess] n_features={len(feat_cols_clean)}')
    return x_train, uid_train_die, y_train_unit_s, y_train_die, feat_cols_clean


def _load_from_precomputed(d):
    # pp.npy 캐시(00_precompute_pp.py). y_die는 이미 broadcast된 die-level 타깃. 표준 모듈과 byte-identical
    man = json.loads((d / 'manifest.json').read_text(encoding='utf-8'))
    assert man.get('clip_y_extreme') == CLIP_Y_EXTREME, f"clip mismatch {man.get('clip_y_extreme')} != {CLIP_Y_EXTREME}"
    assert man.get('pp_fixed') == PP_FIXED, f"pp_fixed mismatch:\n{man.get('pp_fixed')}\n!=\n{PP_FIXED}"
    F = int(man['n_features'])
    pp = np.load(d / 'pp.npy', mmap_mode='r')
    units = np.load(d / 'units.npy')
    feat = json.loads((d / 'feat_cols.json').read_text(encoding='utf-8'))
    x_train = pp[:, :F]
    uid_train_die = pp[:, F].astype(np.int64)
    y_train_die = np.asarray(pp[:, F + 1], dtype=np.float64)
    y_train_unit_s = pd.Series(units[:, 1].astype(np.float64), index=units[:, 0].astype(np.int64))
    print(f'[precomputed] {d/"pp.npy"} F={F} dies={pp.shape[0]:,} units={len(y_train_unit_s):,}')
    return x_train, uid_train_die, y_train_unit_s, y_train_die, feat


if USE_PRECOMPUTED and (PRECOMPUTED_DIR / 'pp.npy').exists():
    X_train, uid_train_die, y_train_unit_s, y_train_die, feat_cols_clean = _load_from_precomputed(PRECOMPUTED_DIR)
    DATA_SOURCE = 'precomputed_pp_npy'
else:
    X_train, uid_train_die, y_train_unit_s, y_train_die, feat_cols_clean = _load_self_preprocess()
    DATA_SOURCE = 'self_preprocess'

N_FEATURES = len(feat_cols_clean)
print(f'X_train={X_train.shape}  dies={len(uid_train_die):,}  units={len(y_train_unit_s):,}  '
      f'n_features={N_FEATURES}  source={DATA_SOURCE}')
print(f'die zero%={float((y_train_die == 0).mean()) * 100:.1f}  '
      f'unit zero%={float((y_train_unit_s.values == 0).mean()) * 100:.1f}')


[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[CLIP_Y_EXTREME] 1.0 -> 0.097417, n=1
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1031 (56개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1031
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 926개
    컬럼: 1031 → 926 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=30%
  제거: 5개, 잔여: 921개
    컬럼: 926 → 921 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 894개
    컬럼: 921 → 894 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 330개, 잔여: 564개
    컬럼: 894 → 564 (330개 제거)
    DataFrame: (104748, 624)

[결측 indicator] 9개 컬럼 추가 (결측률 >= 5%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,748 / 전체 174,572 행
  1단계 (공간 보간, dist<=6

## 3. Optuna study + objective

5-fold unit CV의 OOF RMSE 최소화. `zit_only` objective를 그대로 미러하되 **tau_pi 없음**, 집계는 **MEAN**.

In [4]:
def _mean_die_to_unit(pred_die, uid_die):
    # die 예측을 같은 unit끼리 평균 → unit 예측 (타깃을 4 die에 복사했으므로 SUM이 아니라 MEAN)
    df = pd.DataFrame({KEY_COL: uid_die, 'pred': pred_die})
    return df.groupby(KEY_COL, sort=False)['pred'].mean().reset_index()


# fold 분할은 SEED로 고정 → 모든 trial이 같은 분할을 공유해야 RMSE 비교가 공정하다.
unique_units = y_train_unit_s.index.values
_kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(_kf.split(unique_units))
print(f'[fold] n_folds={N_FOLDS}  seed={SEED}  units={len(unique_units):,}')


def objective(trial):
    t0 = time.time()
    params = hpo.sample_from_space(trial, EMT_SEARCH)
    model_seed = SEED + int(trial.number)   # trial별 모델 seed (재현성 + trial 간 다양성)
    params.update(random_state=model_seed, n_jobs=N_JOBS, verbose=-1, device='cpu', em_tol=1e-7)
    trial.set_user_attr('model_seed', model_seed)

    fold_oof_rmse = []
    oof_pred_unit = pd.Series(np.nan, index=y_train_unit_s.index, dtype=np.float64)

    for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
        tr_units = unique_units[tr_uidx]
        vl_units = unique_units[vl_uidx]
        tr_mask = np.isin(uid_train_die, tr_units)
        vl_mask = np.isin(uid_train_die, vl_units)

        model = EMTboost(**params)
        model.fit(X_train[tr_mask], y_train_die[tr_mask])      # die-level, 타깃은 unit→die 복사본

        pred_die = model.predict(X_train[vl_mask])             # = q·μ, clip>=0 (논문 eq.41)
        unit_pred_df = _mean_die_to_unit(pred_die, uid_train_die[vl_mask])

        oof_pred_unit.loc[unit_pred_df[KEY_COL].values] = unit_pred_df['pred'].values
        y_vl = y_train_unit_s.loc[unit_pred_df[KEY_COL].values].values
        fold_rmse = float(np.sqrt(np.mean((unit_pred_df['pred'].values - y_vl) ** 2)))
        fold_oof_rmse.append(fold_rmse)

        avg = float(np.mean(fold_oof_rmse))
        trial.report(avg, step=fold_idx)
        if trial.should_prune():
            trial.set_user_attr('pruned_at_fold', fold_idx + 1)
            trial.set_user_attr('elapsed_sec', time.time() - t0)
            trial.set_user_attr('fold_oof_rmse', fold_oof_rmse)
            trial.set_user_attr('partial_val_rmse', avg)
            raise optuna.TrialPruned()

    if oof_pred_unit.isna().any():
        raise RuntimeError('OOF NaN: missing fold predictions')

    oof_rmse = float(np.sqrt(np.mean((oof_pred_unit.values - y_train_unit_s.values) ** 2)))
    elapsed = time.time() - t0
    trial.set_user_attr('elapsed_sec', elapsed)
    trial.set_user_attr('fold_oof_rmse', fold_oof_rmse)
    trial.set_user_attr('val_rmse', oof_rmse)
    trial.set_user_attr('oof_rmse', oof_rmse)
    print(f"trial #{trial.number}: oof={oof_rmse:.9f}  elapsed={elapsed:.0f}s  "
          f"zeta={params['zeta']:.3f}  n_em={params['n_em_iters']}  "
          f"q={getattr(model, 'q_', float('nan')):.3f}")
    return oof_rmse


[fold] n_folds=5  seed=42  units=26,187


In [5]:
storage = RDBStorage(
    url=f'sqlite:///{DB_PATH.as_posix()}',
    engine_kwargs={'connect_args': {'timeout': 120.0}},
    heartbeat_interval=300, grace_period=1800,
    failed_trial_callback=RetryFailedTrialCallback(max_retry=1),
)
sampler = TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS)
pruner = MedianPruner(n_startup_trials=N_STARTUP_TRIALS, n_warmup_steps=2)
study = optuna.create_study(
    study_name=EXP_ID, storage=storage, sampler=sampler, pruner=pruner,
    direction='minimize', load_if_exists=RESUME,
)

# anchor는 신규 study에서만 enqueue (RESUME 중복 방지)
if len(study.trials) == 0:
    study.enqueue_trial(dict(EMT_ANCHOR))
    print(f'[enqueue] zit_only μ anchor ({len(EMT_ANCHOR)} HP) 첫 trial로')
else:
    print(f'[enqueue skip] existing trials={len(study.trials)}')

study_meta = {
    'exp_id': EXP_ID, 'user': USER,
    'model': 'EMTboost (Zhou Qian Yang 2019) — scalar q/phi, mu=LightGBM Tweedie. notebook HPO',
    'paper': 'Tweedie Gradient Boosting for Extremely Unbalanced Zero-inflated Data (2019, arXiv:1811.10192)',
    'die_to_unit': 'zit_only-style: unit target broadcast to 4 dies, die pred MEAN-aggregated, NO tau_pi (scalar pi)',
    'predict': 'E[Y]=q*mu (eq.41)',
    'n_folds': N_FOLDS, 'n_jobs': N_JOBS,
    'pp_fixed': PP_FIXED, 'clip_y_extreme': CLIP_Y_EXTREME, 'n_features': N_FEATURES,
    'data_source': DATA_SOURCE,
    'search_space': EMT_SEARCH, 'anchor': EMT_ANCHOR,
    'anchor_source': 'zit-only-final-003 best #24 (mu params + zeta/n_em)',
    'sampler': 'TPE seed=None multivariate group',
    'pruner': f'MedianPruner n_startup={N_STARTUP_TRIALS} n_warmup=2',
    'seed': SEED, 'model_random_state': 'SEED + trial.number',
}
for k, v in study_meta.items():
    study.set_user_attr(k, str(v))
print('study', study.study_name, ' existing_trials', len(study.trials))


[enqueue] zit_only μ anchor (11 HP) 첫 trial로
study emt-boost-hp-001  existing_trials 1


## 4. 튜닝 실행 (마감까지)

In [6]:
if END_AT:
    end_dt = datetime.fromisoformat(END_AT)
    remaining = (end_dt - datetime.now()).total_seconds()
    if remaining <= 0:
        print(f'[end-at] {end_dt.isoformat()} 이미 지남 → TIMEOUT_HOURS={TIMEOUT_HOURS}h로 대체')
        timeout = None if TIMEOUT_HOURS <= 0 else TIMEOUT_HOURS * 3600
    else:
        timeout = remaining
        print(f'[end-at] target={end_dt.isoformat()} → 남은 시간 {timeout/3600:.2f}h')
else:
    timeout = None if TIMEOUT_HOURS <= 0 else TIMEOUT_HOURS * 3600
    print(f'[timeout] {TIMEOUT_HOURS}h')

t_start = time.time()
study.optimize(objective, n_trials=N_TRIALS, timeout=timeout, n_jobs=1, show_progress_bar=False)
print(f'[done] elapsed={(time.time() - t_start) / 3600:.2f}h  total_trials={len(study.trials)}')
try:
    print(f'best_value={study.best_value:.9f}  best_trial=#{study.best_trial.number}')
except ValueError:
    print('완료된 trial 없음')


[end-at] target=2026-06-07T08:00:00 → 남은 시간 9.35h
trial #0: oof=0.005590516  elapsed=577s  zeta=1.120  n_em=16  q=0.458
trial #1: oof=0.005590914  elapsed=442s  zeta=1.182  n_em=28  q=0.369
trial #2: oof=0.005581292  elapsed=543s  zeta=1.161  n_em=23  q=0.392
trial #3: oof=0.005581984  elapsed=658s  zeta=1.059  n_em=13  q=0.563
trial #4: oof=0.005593551  elapsed=385s  zeta=1.194  n_em=28  q=0.364
trial #5: oof=0.005576005  elapsed=488s  zeta=1.018  n_em=10  q=0.573
trial #6: oof=0.005586639  elapsed=757s  zeta=1.158  n_em=26  q=0.388
trial #7: oof=0.005586413  elapsed=841s  zeta=1.126  n_em=19  q=0.449
trial #8: oof=0.005592264  elapsed=440s  zeta=1.163  n_em=13  q=0.384
trial #9: oof=0.005590725  elapsed=480s  zeta=1.145  n_em=30  q=0.409
trial #10: oof=0.005594594  elapsed=440s  zeta=1.211  n_em=16  q=0.358
trial #11: oof=0.005585662  elapsed=611s  zeta=1.040  n_em=14  q=0.652
trial #12: oof=0.005592656  elapsed=460s  zeta=1.051  n_em=10  q=0.525
trial #13: oof=0.005579273  elapsed=5

## 5. 결과 · best params 저장

In [ ]:
df = study.trials_dataframe()
n_complete = int((df['state'] == 'COMPLETE').sum()) if len(df) else 0
n_pruned = int((df['state'] == 'PRUNED').sum()) if len(df) else 0
print(f'COMPLETE={n_complete}  PRUNED={n_pruned}  total={len(df)}')

if n_complete:
    done = df[df['state'] == 'COMPLETE'].sort_values('value')
    show_cols = [c for c in ['number', 'value', 'user_attrs_oof_rmse', 'user_attrs_elapsed_sec',
                             'params_zeta', 'params_n_em_iters', 'params_mu_n_estimators',
                             'params_mu_learning_rate', 'params_mu_max_depth'] if c in done.columns]
    try:
        display(done[show_cols].head(20))
    except NameError:
        print(done[show_cols].head(20).to_string())

    best = study.best_trial
    best_params_resolved = dict(best.params)        # zeta, n_em_iters, mu_*
    out_json = {
        'exp_id': EXP_ID,
        'model_name': 'emt_boost',
        'source_db': str(DB_PATH),
        'source_study_name': EXP_ID,
        'best_trial_number': best.number,
        'best_trial_state': str(best.state),
        'best_oof_rmse': best.user_attrs.get('oof_rmse', best.value),
        'best_params_resolved': best_params_resolved,
        'best_tau_pi': None,                        # EMTboost: tau_pi 없음 (scalar pi)
        'effective_pp_params': PP_FIXED,
        'feature_names': None,
        'n_features': N_FEATURES,
        'study_meta': dict(study.user_attrs),
    }
    best_json_path = OUT_DIR / f'best_params_{EXP_ID}_t{best.number}.json'
    best_json_path.write_text(json.dumps(out_json, indent=2, ensure_ascii=False), encoding='utf-8')
    print('saved ->', best_json_path)
    print(json.dumps(best_params_resolved, indent=2, ensure_ascii=False))
else:
    print('완료된 trial이 없어 저장 생략.')

COMPLETE=41  PRUNED=0  total=41


,number,value,user_attrs_oof_rmse,user_attrs_elapsed_sec,params_zeta,params_n_em_iters,params_mu_n_estimators,params_mu_learning_rate,params_mu_max_depth
31,31,0.005544,0.005544,1286.525090,1.079237,28,228,0.004811,8
26,26,0.005545,0.005545,1145.247979,1.061016,27,248,0.004757,6
30,30,0.005546,0.005546,1246.670681,1.035239,25,237,0.003835,7
34,34,0.005546,0.005546,1243.956089,1.012783,26,213,0.004039,8
27,27,0.005547,0.005547,1200.610151,1.031892,28,244,0.004158,6
25,25,0.005548,0.005548,1191.461228,1.058710,26,202,0.004135,7
29,29,0.005550,0.005550,1241.548207,1.100363,25,280,0.004177,7
32,32,0.005550,0.005550,1361.272435,1.089662,29,256,0.003917,8
33,33,0.005552,0.005552,1072.900415,1.033198,27,250,0.004110,5
39,39,0.005554,0.005554,1074.468020,1.011520,22,232,0.003294,8


saved -> C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\01_zit\emt_boost\hp\001\best_params_emt-boost-hp-001_t31.json
{
  "zeta": 1.0792374959955573,
  "n_em_iters": 28,
  "mu_n_estimators": 228,
  "mu_learning_rate": 0.004811227720104855,
  "mu_num_leaves": 43,
  "mu_max_depth": 8,
  "mu_min_child_samples": 293,
  "mu_subsample": 0.7414643490900497,
  "mu_colsample_bytree": 0.3770948099450344,
  "mu_reg_alpha": 0.00463113035299218,
  "mu_reg_lambda": 0.00011319430507754325
}
